In [1]:
import pandas as pd 
import numpy as np 
import polars as pl
from typing import List
from collections import Counter
import glob, os

import yaml
with open("../../src/configs.yml", "r") as configs:
    configs = yaml.safe_load(configs)

DATAPATH = configs["SERVER_DATA"]
ROOTPATH = configs["SERVER_WD"]

In [2]:
# get matched subreddits
input_fpath = f"{DATAPATH}/data/samples"
X0 = 10; N0 = 1
X1 = 20; N1 = 10
X2 = "inf"; N2 = 1
covs1 = "lifetime_d"
covs2 = "lifetime_d_n_post_author_till2019_log"
matched1_indp_df = pd.read_csv(f"{input_fpath}/cem-batch4-new/matched_table_banned_3274_cem_X{X0}N{N0}_{covs1}.csv")
matched2_indp_df = pd.read_csv(f"{input_fpath}/cem-batch4-new/matched_table_banned_3274_cem_X{X1}N{N1}_{covs2}.csv")
matched2_conn_df = pd.read_csv(f"{input_fpath}/cem-batch4-new/matched_table_banned_3274_cem_X{X2}N{N2}_{covs2}.csv")

# get banned subreddits and random samples
samples_df = pd.read_csv(f"{DATAPATH}/data/samples/to_match_table_selfid_banned_3274_260528.csv")
samples_df = samples_df[samples_df["lifetime_d"]>0]

banned_df = samples_df[samples_df["if_banned"]==1]
random_df = samples_df[samples_df["if_banned"]==0].sample(n=len(banned_df), replace=False, random_state=42)

# banned_random_df = pd.concat([banned_df, random_df])
# banned_random_df.to_csv(f"{DATAPATH}/data/samples/sample_banned_random_3274_250629.csv", index=False)

matched1_indp = set(matched1_indp_df[matched1_indp_df["if_banned"]==0]["subreddit"].tolist())
matched2_indp = set(matched2_indp_df[matched2_indp_df["if_banned"]==0]["subreddit"].tolist())
matched2_conn = set(matched2_conn_df[matched2_conn_df["if_banned"]==0]["subreddit"].tolist())
random = set(random_df["subreddit"].tolist())
banned = set(banned_df["subreddit"].tolist())
print(len(matched1_indp), len(matched2_indp), len(matched2_conn), len(random), len(banned))

3274 3274 3274 3274 3274


In [3]:
from pathlib import Path

output_path = "../../data-public/subreddit_sets.csv"

subreddit_sets = {
    "matched1_indp": matched1_indp,
    "matched2_indp": matched2_indp,
    "matched2_conn": matched2_conn,
    "random": random,
    "banned": banned,
}

# Sort names within each set for a stable, readable CSV.
sets_df = pd.DataFrame(
    [
        {"set": name, "subreddit": subreddit}
        for name, subreddits in subreddit_sets.items()
        for subreddit in sorted(subreddits)
    ],
    columns=["set", "subreddit"],
)

sets_df.to_csv(output_path, index=False)

In [4]:
# loading functions
sets_df = pd.read_csv("../../data-public/subreddit_sets.csv")

subreddit_sets = {
    name: set(group["subreddit"])
    for name, group in sets_df.groupby("set")
}

matched1_indp_loaded = subreddit_sets["matched1_indp"]
matched2_indp_loaded= subreddit_sets["matched2_indp"]
matched2_conn_loaded = subreddit_sets["matched2_conn"]
random_loaded = subreddit_sets["random"]
banned_loaded = subreddit_sets["banned"]

assert matched1_indp == matched1_indp_loaded
assert matched2_indp == matched2_indp_loaded
assert matched2_conn == matched2_conn_loaded
assert random == random_loaded
assert banned == banned_loaded